# Weld Seam Defect Detection with YOLOv8m

Source: [ML_Weld_Seam on Kaggle](https://www.kaggle.com/code/youssefsalem05/ml-weld-seam), version 1.

This portfolio copy replaces embedded experiment-tracking authentication with an optional environment variable. See README.md for dataset, GPU, and network setup. Training outputs and model weights are not included.


In [ ]:
# Install YOLOv8 and necessary dependencies
!pip install ultralytics --index-url https://pypi.org/simple/

import os
import shutil
import matplotlib.pyplot as plt
import cv2
import random
from IPython.display import Image, display
import pandas as pd
import seaborn as sns
from ultralytics import YOLO

In [ ]:
# Optional experiment tracking. Set WANDB_API_KEY outside this notebook.
# Never put API keys directly in a notebook or commit them to version control.
import wandb

wandb_api_key = os.environ.get("WANDB_API_KEY")
if wandb_api_key:
    wandb.login(key=wandb_api_key)
else:
    os.environ["WANDB_MODE"] = "disabled"


In [ ]:
root_dir = '/kaggle/input/datasets/sukmaadhiwijaya/welding-defect-object-detection'
v1_dir = os.path.join(root_dir, 'The Welding Defect Dataset/The Welding Defect Dataset')
v2_dir = os.path.join(root_dir, 'The Welding Defect Dataset - v2/The Welding Defect Dataset - v2')

# Verify
print("v1 exists:", os.path.exists(v1_dir))
print("v2 exists:", os.path.exists(v2_dir))
print("\nv1 contents:", os.listdir(v1_dir))
print("v2 contents:", os.listdir(v2_dir))

In [ ]:
# Ensure the YOLOv8 data directories exist
os.makedirs('yolov8/data/welding_v1', exist_ok=True)
os.makedirs('yolov8/data/welding_v2', exist_ok=True)

In [ ]:
# Function to display images
def display_images(image_dir, num_images=9):
    images = os.listdir(image_dir)
    sample_images = random.sample(images, num_images)
    plt.figure(figsize=(15, 15))
    for i, image_name in enumerate(sample_images):
        image_path = os.path.join(image_dir, image_name)
        img = cv2.imread(image_path)
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        plt.subplot(3, 3, i + 1)
        plt.imshow(img_rgb)
        plt.axis('off')
    plt.show()

# Display some training images
print("Displaying some training images from version 1:")
display_images(os.path.join(v1_dir, 'train/images'))

print("Displaying some training images from version 2:")
display_images(os.path.join(v2_dir, 'train/images'))

In [ ]:
# Copy dataset to YOLOv8 directory
def copy_dataset_to_yolo(src_dir, dest_dir):
    if os.path.exists(dest_dir):
        shutil.rmtree(dest_dir)
    shutil.copytree(src_dir, dest_dir)

# Copy version 1 dataset
copy_dataset_to_yolo(v1_dir, 'yolov8/data/welding_v1')

# Copy version 2 dataset
copy_dataset_to_yolo(v2_dir, 'yolov8/data/welding_v2')

In [ ]:
# Update data.yaml paths
def update_data_yaml(data_yaml_path, base_dir):
    with open(data_yaml_path, 'r') as file:
        data_yaml = file.read()
    data_yaml = data_yaml.replace('../train/images', os.path.join(base_dir, 'train/images'))
    data_yaml = data_yaml.replace('../valid/images', os.path.join(base_dir, 'valid/images'))
    data_yaml = data_yaml.replace('../test/images', os.path.join(base_dir, 'test/images'))
    with open(data_yaml_path, 'w') as file:
        file.write(data_yaml)

# Update data.yaml for version 1
update_data_yaml('yolov8/data/welding_v1/data.yaml', '/kaggle/working/yolov8/data/welding_v1')

# Update data.yaml for version 2
update_data_yaml('yolov8/data/welding_v2/data.yaml', '/kaggle/working/yolov8/data/welding_v2')

# Print and verify updated data.yaml files
def print_data_yaml(data_yaml_path):
    with open(data_yaml_path, 'r') as file:
        print(file.read())

print("Updated data.yaml for version 1:")
print_data_yaml('yolov8/data/welding_v1/data.yaml')

print("Updated data.yaml for version 2:")
print_data_yaml('yolov8/data/welding_v2/data.yaml')

In [ ]:
# EDA and Preprocessing
def load_labels(label_dir):
    label_files = os.listdir(label_dir)
    data = []
    for label_file in label_files:
        with open(os.path.join(label_dir, label_file), 'r') as file:
            lines = file.readlines()
            for line in lines:
                class_id, x_center, y_center, width, height = map(float, line.strip().split())
                data.append([label_file, class_id, x_center, y_center, width, height])
    df = pd.DataFrame(data, columns=['file', 'class_id', 'x_center', 'y_center', 'width', 'height'])
    return df

v1_train_labels = load_labels(os.path.join(v1_dir, 'train/labels'))
v2_train_labels = load_labels(os.path.join(v2_dir, 'train/labels'))

In [ ]:
# Plot class distribution
plt.figure(figsize=(12, 6))
sns.countplot(data=v1_train_labels, x='class_id')
plt.title('Class Distribution in Training Set (Version 1)')
plt.show()


In [ ]:
plt.figure(figsize=(12, 6))
sns.countplot(data=v2_train_labels, x='class_id')
plt.title('Class Distribution in Training Set (Version 2)')
plt.show()

In [ ]:
# Train YOLOv8 model with augmentation and hyperparameter tuning
# Initialize model
model_v1 = YOLO('yolov8m.pt')  # YOLOv8m (medium) model
model_v2 = YOLO('yolov8m.pt')  # YOLOv8m (medium) model

In [ ]:
# Train model on version 1 dataset
model_v1.train(
    data='/kaggle/working/yolov8/data/welding_v1/data.yaml',
    epochs=100,
    imgsz=640,
    batch=16,
    name='welding_v1_aug',
    device='0,1',   
    patience=20,
    cache=True,
    # Learning rate
    lr0=0.01,
    lrf=0.01,
    momentum=0.937,
    weight_decay=0.0005,
    # Warmup
    warmup_epochs=3.0,
    warmup_momentum=0.8,
    warmup_bias_lr=0.1,
    # Loss weights (fixed box from 0.05 → 7.5)
    box=7.5,
    cls=0.5,
    iou=0.5,
    # Augmentation
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    translate=0.1,
    scale=0.5,
    mosaic=1.0,
    mixup=0.2,        # toned down from 0.5
    # Logging
    save_period=10,
    plots=True,
)

# Train model on version 2 dataset
model_v2.train(
    data='/kaggle/working/yolov8/data/welding_v2/data.yaml',
    epochs=100,
    imgsz=640,
    batch=16,
    name='welding_v2_aug',
    device='0,1',   
    patience=20,
    cache=True,
    # Learning rate
    lr0=0.01,
    lrf=0.01,
    momentum=0.937,
    weight_decay=0.0005,
    # Warmup
    warmup_epochs=3.0,
    warmup_momentum=0.8,
    warmup_bias_lr=0.1,
    # Loss weights (fixed box from 0.05 → 7.5)
    box=7.5,
    cls=0.5,
    iou=0.5,
    # Augmentation
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    translate=0.1,
    scale=0.5,
    mosaic=1.0,
    mixup=0.2,
    # Logging
    save_period=10,
    plots=True,
)

In [ ]:
import os
import random
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display

# ==============================
# 1. CONFIG (EDIT ONLY THIS)
# ==============================

v1_dir = '/kaggle/working/yolov8/data/welding_v1'
v2_dir = '/kaggle/working/yolov8/data/welding_v2'

data_yaml_v1 = os.path.join(v1_dir, 'data.yaml')
data_yaml_v2 = os.path.join(v2_dir, 'data.yaml')

test_imgs_v1 = os.path.join(v1_dir, 'test/images')
test_imgs_v2 = os.path.join(v2_dir, 'test/images')


# ==============================
# 2. SANITY CHECK (VERY IMPORTANT)
# ==============================

def check_paths():
    paths = [
        data_yaml_v1, data_yaml_v2,
        test_imgs_v1, test_imgs_v2
    ]
    
    print("\n🔍 Checking paths...")
    for p in paths:
        print(f"{p} → {'✅ EXISTS' if os.path.exists(p) else '❌ MISSING'}")

check_paths()


# ==============================
# 3. EVALUATION FUNCTION
# ==============================

def evaluate_model(model, data_yaml, img_dir, model_name, conf=0.25, iou=0.5, num_images=5):

    print(f"\n{'='*60}")
    print(f"  Evaluating: {model_name}")
    print(f"{'='*60}")

    # ---- A. Metrics ----
    metrics = model.val(
        data=data_yaml,   # ✅ FIX
        conf=conf,
        iou=iou,
        verbose=False
    )

    print("\n📊 Core Metrics:")
    print(f"mAP@0.5       : {metrics.box.map50:.4f}")
    print(f"mAP@0.5:0.95  : {metrics.box.map:.4f}")
    print(f"Precision     : {metrics.box.mp:.4f}")
    print(f"Recall        : {metrics.box.mr:.4f}")

    # ---- B. Per-class ----
    class_names = model.names
    per_class = []

    for i, name in class_names.items():
        per_class.append({
            "Class": name,
            "AP@0.5": metrics.box.ap50[i] if i < len(metrics.box.ap50) else 0,
            "AP@0.5:0.95": metrics.box.ap[i] if i < len(metrics.box.ap) else 0
        })

    df = pd.DataFrame(per_class)
    print("\n📋 Per-class performance:")
    print(df.to_string(index=False))

    # ---- C. Visualization ----
    print("\n🖼️ Sample Predictions:")

    images = [f for f in os.listdir(img_dir) if f.endswith(('.jpg','.png','.jpeg'))]
    sample = random.sample(images, min(num_images, len(images)))

    save_dir = f'/kaggle/working/{model_name}_preds'
    os.makedirs(save_dir, exist_ok=True)

    for img in sample:
        path = os.path.join(img_dir, img)
        results = model(path, conf=conf, iou=iou)

        for r in results:
            out_path = os.path.join(save_dir, img)
            r.save(filename=out_path)
            display(Image(filename=out_path))

    return metrics, df


# ==============================
# 4. RUN EVALUATION
# ==============================

metrics_v1, df_v1 = evaluate_model(
    model_v1,
    data_yaml_v1,
    test_imgs_v1,
    "YOLOv8m_v1"
)

metrics_v2, df_v2 = evaluate_model(
    model_v2,
    data_yaml_v2,
    test_imgs_v2,
    "YOLOv8m_v2"
)


# ==============================
# 5. MODEL COMPARISON
# ==============================

print(f"\n{'='*60}")
print("📊 FINAL COMPARISON")
print(f"{'='*60}")

comparison = pd.DataFrame({
    "Metric": ["mAP@0.5", "mAP@0.5:0.95", "Precision", "Recall"],
    "V1": [
        metrics_v1.box.map50,
        metrics_v1.box.map,
        metrics_v1.box.mp,
        metrics_v1.box.mr
    ],
    "V2": [
        metrics_v2.box.map50,
        metrics_v2.box.map,
        metrics_v2.box.mp,
        metrics_v2.box.mr
    ]
})

comparison["Winner"] = comparison.apply(
    lambda r: "V1 ✅" if r["V1"] > r["V2"] else "V2 ✅",
    axis=1
)

print(comparison.to_string(index=False))


# ==============================
# 6. VISUAL COMPARISON PLOT
# ==============================

x = np.arange(len(comparison["Metric"]))
width = 0.35

plt.figure(figsize=(10,5))

plt.bar(x - width/2, comparison["V1"], width, label="V1")
plt.bar(x + width/2, comparison["V2"], width, label="V2")

plt.xticks(x, comparison["Metric"])
plt.ylim(0,1)
plt.title("Model Comparison")
plt.ylabel("Score")
plt.legend()

for i, v in enumerate(comparison["V1"]):
    plt.text(i - width/2, v + 0.01, f"{v:.3f}", ha='center')

for i, v in enumerate(comparison["V2"]):
    plt.text(i + width/2, v + 0.01, f"{v:.3f}", ha='center')

plt.tight_layout()
plt.show()